# Predictions02 - Using v2 Models (Enhanced Features)
Loads models trained in **Notebook03** with 24 features:
- H2H last 5 matches
- Recent form (last 5) — avg scored/conceded only
- Home vs Away win rate split
- Competition importance (1-4)
- FIFA ranking & rank difference

In [ ]:
# ============================================================
# CELL 1 - Load Data & Normalize Country Names
# ============================================================
import pandas as pd
import numpy as np
import joblib

results      = pd.read_csv('../../data/results.csv')
shootouts    = pd.read_csv('../../data/shootouts.csv')
former_names = pd.read_csv('../../data/former_names.csv')
rankings     = pd.read_csv('../../dataII/rankings.csv')

results['date']   = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])

name_map = dict(zip(former_names['former'], former_names['current']))
def normalize(name): return name_map.get(name, name)

results['home_team']   = results['home_team'].apply(normalize)
results['away_team']   = results['away_team'].apply(normalize)
shootouts['home_team'] = shootouts['home_team'].apply(normalize)
shootouts['away_team'] = shootouts['away_team'].apply(normalize)
shootouts['winner']    = shootouts['winner'].apply(normalize)

results_sorted = results.dropna(subset=['home_score','away_score']).sort_values('date').reset_index(drop=True)

rank_lookup   = dict(zip(rankings['team'], rankings['fifa_rank']))
points_lookup = dict(zip(rankings['team'], rankings['fifa_points']))

print('Data loaded!')
print('results:', results_sorted.shape, '  rankings:', rankings.shape)


In [2]:
# ============================================================
# CELL 2 - Helper Functions (same as Notebook03)
# ============================================================

def get_rank(team):
    return rank_lookup.get(team, 200)

def team_stats(team, before_date, n=20):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        ((results_sorted['home_team'] == team) | (results_sorted['away_team'] == team))
    ].tail(n)
    if len(past) == 0:
        return {'win_rate': 0.5, 'avg_scored': 1.0, 'avg_conceded': 1.0, 'n_matches': 0}
    wins, scored, conceded = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == team:
            wins.append(1 if m['home_score'] > m['away_score'] else 0)
            scored.append(m['home_score']); conceded.append(m['away_score'])
        else:
            wins.append(1 if m['away_score'] > m['home_score'] else 0)
            scored.append(m['away_score']); conceded.append(m['home_score'])
    return {'win_rate': np.mean(wins), 'avg_scored': np.mean(scored),
            'avg_conceded': np.mean(conceded), 'n_matches': len(past)}

def team_stats_split(team, before_date, n=20):
    past_home = results_sorted[
        (results_sorted['date'] < before_date) & (results_sorted['home_team'] == team)
    ].tail(n)
    past_away = results_sorted[
        (results_sorted['date'] < before_date) & (results_sorted['away_team'] == team)
    ].tail(n)
    home_wr = (past_home['home_score'] > past_home['away_score']).mean() if len(past_home) > 0 else 0.5
    away_wr = (past_away['away_score'] > past_away['home_score']).mean() if len(past_away) > 0 else 0.5
    return float(home_wr), float(away_wr)

def recent_form(team, before_date, n=5):
    s = team_stats(team, before_date, n=n)
    return s['win_rate'], s['avg_scored'], s['avg_conceded']

def h2h_stats(home_team, away_team, before_date, n=5):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        (
            ((results_sorted['home_team'] == home_team) & (results_sorted['away_team'] == away_team)) |
            ((results_sorted['home_team'] == away_team) & (results_sorted['away_team'] == home_team))
        )
    ].tail(n)
    if len(past) == 0:
        return {'h2h_home_win_rate': 0.5, 'h2h_avg_home_scored': 1.0, 'h2h_avg_away_scored': 1.0, 'h2h_n': 0}
    home_wins, home_scored, away_scored = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == home_team:
            home_wins.append(1 if m['home_score'] > m['away_score'] else 0)
            home_scored.append(m['home_score']); away_scored.append(m['away_score'])
        else:
            home_wins.append(1 if m['away_score'] > m['home_score'] else 0)
            home_scored.append(m['away_score']); away_scored.append(m['home_score'])
    return {
        'h2h_home_win_rate':   np.mean(home_wins),
        'h2h_avg_home_scored': np.mean(home_scored),
        'h2h_avg_away_scored': np.mean(away_scored),
        'h2h_n':               len(past)
    }

def comp_importance(tournament):
    t = str(tournament).lower()
    if 'fifa world cup' in t:                                              return 4
    if any(x in t for x in ['euro', 'copa america', 'asian cup',
                              'africa cup', 'gold cup', 'nations cup']):   return 3
    if any(x in t for x in ['qualifier', 'qualification']):               return 2
    return 1

def shootout_win_rate(team, before_date):
    past = shootouts[
        (shootouts['date'] < before_date) &
        ((shootouts['home_team'] == team) | (shootouts['away_team'] == team))
    ]
    if len(past) == 0: return 0.5
    return (past['winner'] == team).sum() / len(past)

print('Helper functions ready!')

Helper functions ready!


In [ ]:
# ============================================================
# CELL 3 - Load v2 Models
# ============================================================
model_outcome    = joblib.load('../../dataII/V2/model_outcome_v2.pkl')
model_home_score = joblib.load('../../dataII/V2/model_home_score_v2.pkl')
model_away_score = joblib.load('../../dataII/V2/model_away_score_v2.pkl')

FEATURES = [
    'is_neutral', 'year', 'comp_importance',
    'home_win_rate', 'home_avg_scored', 'home_avg_conceded',
    'home_home_win_rate', 'home_away_win_rate',
    'home_form_avg_scored', 'home_form_avg_conceded',
    'away_win_rate', 'away_avg_scored', 'away_avg_conceded',
    'away_home_win_rate', 'away_away_win_rate',
    'away_form_avg_scored', 'away_form_avg_conceded',
    'h2h_home_win_rate', 'h2h_avg_home_scored', 'h2h_avg_away_scored', 'h2h_n',
    'home_fifa_rank', 'away_fifa_rank', 'rank_diff',
]

print(f'Loaded: {type(model_outcome).__name__} (outcome)')
print(f'Loaded: {type(model_home_score).__name__} (home score)')
print(f'Loaded: {type(model_away_score).__name__} (away score)')
print(f'Feature count: {len(FEATURES)}')


In [4]:
# ============================================================
# CELL 4 - Predict Function (v2) — with neutral symmetry fix
# ============================================================

def _build_row(home_team, away_team, is_neutral, tournament):
    """Build a single feature row. Always treats first arg as 'home slot'."""
    today = pd.Timestamp.today()
    h           = team_stats(home_team, today)
    a           = team_stats(away_team, today)
    h_hw, h_aw  = team_stats_split(home_team, today)
    a_hw, a_aw  = team_stats_split(away_team, today)
    _, h_fsc, h_fcc = recent_form(home_team, today)
    _, a_fsc, a_fcc = recent_form(away_team, today)
    h2h         = h2h_stats(home_team, away_team, today)
    home_rank   = get_rank(home_team)
    away_rank   = get_rank(away_team)
    return pd.DataFrame([{
        'is_neutral':             is_neutral,
        'year':                   today.year,
        'comp_importance':        comp_importance(tournament),
        'home_win_rate':          h['win_rate'],
        'home_avg_scored':        h['avg_scored'],
        'home_avg_conceded':      h['avg_conceded'],
        'home_home_win_rate':     h_hw,
        'home_away_win_rate':     h_aw,
        'home_form_avg_scored':   h_fsc,
        'home_form_avg_conceded': h_fcc,
        'away_win_rate':          a['win_rate'],
        'away_avg_scored':        a['avg_scored'],
        'away_avg_conceded':      a['avg_conceded'],
        'away_home_win_rate':     a_hw,
        'away_away_win_rate':     a_aw,
        'away_form_avg_scored':   a_fsc,
        'away_form_avg_conceded': a_fcc,
        'h2h_home_win_rate':      h2h['h2h_home_win_rate'],
        'h2h_avg_home_scored':    h2h['h2h_avg_home_scored'],
        'h2h_avg_away_scored':    h2h['h2h_avg_away_scored'],
        'h2h_n':                  h2h['h2h_n'],
        'home_fifa_rank':         home_rank,
        'away_fifa_rank':         away_rank,
        'rank_diff':              away_rank - home_rank,
    }])


def predict(team_a, team_b, is_neutral=0, tournament='Friendly'):
    """
    Predict match outcome and score.

    For neutral venues (is_neutral=1): runs prediction in BOTH directions
    (A vs B and B vs A) then averages — so order of teams doesn't affect result.

    For real home matches (is_neutral=0): team_a is the home team, order matters.
    """
    label_map = {0: 'Home Win', 1: 'Draw', 2: 'Away Win'}

    row_ab = _build_row(team_a, team_b, is_neutral, tournament)

    if is_neutral:
        # Run both orderings
        row_ba = _build_row(team_b, team_a, is_neutral, tournament)

        proba_ab = model_outcome.predict_proba(row_ab)[0]  # [p_A_win, p_draw, p_B_win]
        proba_ba = model_outcome.predict_proba(row_ba)[0]  # [p_B_win, p_draw, p_A_win]

        # Flip ba so it's back in A's perspective: [p_A_win, p_draw, p_B_win]
        proba_ba_flipped = np.array([proba_ba[2], proba_ba[1], proba_ba[0]])

        # Average the two perspectives
        proba = (proba_ab + proba_ba_flipped) / 2

        # Average predicted scores
        a_goals = (model_home_score.predict(row_ab)[0] + model_away_score.predict(row_ba)[0]) / 2
        b_goals = (model_away_score.predict(row_ab)[0] + model_home_score.predict(row_ba)[0]) / 2
    else:
        proba   = model_outcome.predict_proba(row_ab)[0]
        a_goals = model_home_score.predict(row_ab)[0]
        b_goals = model_away_score.predict(row_ab)[0]

    a_goals = max(0, int(round(a_goals)))
    b_goals = max(0, int(round(b_goals)))
    outcome = np.argmax(proba)

    venue = 'Neutral' if is_neutral else f'{team_a} home ground'
    print(f'  {team_a}  vs  {team_b}  [{venue}]')
    print(f'  Predicted Score: {team_a} {a_goals} - {b_goals} {team_b}')
    print(f'  Outcome        : {label_map[outcome]}')
    print(f'  {team_a} win  : {proba[0]*100:.1f}%')
    print(f'  Draw           : {proba[1]*100:.1f}%')
    print(f'  {team_b} win  : {proba[2]*100:.1f}%')
    print()

print('predict() ready — neutral symmetry fix applied!')


predict() ready — neutral symmetry fix applied!


In [5]:
# Match Start 29 - June KHT 
predict('South Africa', 'Canada', is_neutral=1, tournament='FIFA World Cup')
predict('Canada', 'South Africa', is_neutral=0, tournament='FIFA World Cup')

# predict('Brasil', 'Japann', is_neutral=1, tournament='FIFA World Cup')

  South Africa  vs  Canada  [Neutral]
  Predicted Score: South Africa 1 - 1 Canada
  Outcome        : Draw
  South Africa win  : 31.0%
  Draw           : 35.7%
  Canada win  : 33.3%

  Canada  vs  South Africa  [Canada home ground]
  Predicted Score: Canada 2 - 1 South Africa
  Outcome        : Home Win
  Canada win  : 40.7%
  Draw           : 36.0%
  South Africa win  : 23.4%



In [6]:
# predict('Brasil', 'Japann', is_neutral=1, tournament='FIFA World Cup')
# predict('Brasil', 'Japann', is_neutral=1, tournament='FIFA World Cup')